In [ ]:
#!/usr/bin/env python3
from pathlib import Path
import re

in_gfa = Path("/home/jpereira/AAGrafos/entrega/fastas/grch38_hprc_r2.minigraph.gfa")
out_gfa = Path("/home/jpereira/AAGrafos/entrega/fastas/grch38_hprc_r2.minigraph.gfa")

seg_map = {}

# First pass: collect segment IDs from S lines
with in_gfa.open() as f:
    for line in f:
        if not line.startswith("S\t"):
            continue
        fields = line.rstrip("\n").split("\t")
        seg_id = fields[1]
        m = re.fullmatch(r"s(\d+)", seg_id)
        if m:
            seg_map[seg_id] = m.group(1)
        elif seg_id.isdigit():
            seg_map[seg_id] = seg_id
        else:
            raise ValueError(f"Unsupported segment ID format: {seg_id}")

def convert_oriented_token(token: str) -> str:
    # Examples: s10+, s10-, 10+, 10-
    m = re.fullmatch(r"(.+?)([+-])", token)
    if not m:
        return seg_map.get(token, token)
    seg, orient = m.groups()
    return f"{seg_map.get(seg, seg)}{orient}"

with in_gfa.open() as fin, out_gfa.open("w") as fout:
    for line in fin:
        line = line.rstrip("\n")
        if not line:
            fout.write("\n")
            continue

        fields = line.split("\t")
        rec_type = fields[0]

        if rec_type == "S":
            fields[1] = seg_map[fields[1]]

        elif rec_type == "L":
            # L <from> <from_orient> <to> <to_orient> <overlap> ...
            fields[1] = seg_map.get(fields[1], fields[1])
            fields[3] = seg_map.get(fields[3], fields[3])

        elif rec_type == "P":
            # P <name> <seg1+,seg2-,...> <cigar,...>
            segs = fields[2].split(",")
            fields[2] = ",".join(convert_oriented_token(s) for s in segs)

        elif rec_type == "W":
            # W <sample> <hap> <seqname> <start> <end> <walk>
            # walk format is like >s1>s2<s3 ...
            walk = fields[6]

            def repl(m):
                direction = m.group(1)
                seg = m.group(2)
                return direction + seg_map.get(seg, seg)

            walk = re.sub(r'([<>])([^<>]+)', repl, walk)
            fields[6] = walk

        fout.write("\t".join(fields) + "\n")

print(f"Wrote fixed GFA to: {out_gfa}")
print(f"Total segments mapped: {len(seg_map)}")